# Neural Decoding of Music from EEG

**Reproduction of Daly (2023)**  
*Scientific Reports*, DOI: [10.1038/s41598-022-27361-x](https://doi.org/10.1038/s41598-022-27361-x)

This notebook demonstrates the full analysis pipeline:
1. Data loading
2. EEG preprocessing
3. fMRI GLM analysis and dipole selection
4. eLoreta source localisation and feature extraction
5. biLSTM training and inference
6. Evaluation (r_time, r_freq, SSIM, rank accuracy)

Set `DATA_DIR` below to point to your downloaded dataset.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Paths — adjust to your local dataset location
DATA_DIR = PROJECT_ROOT / "data" / "ds002691"   # EEG-fMRI dataset
OUTPUT_DIR = PROJECT_ROOT / "results" / "demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir   : {DATA_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Data exists: {DATA_DIR.exists()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mne

mne.set_log_level('WARNING')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 1. Data Loading

Load EEG and audio data for one subject.

In [ ]:
from src.data_loader import (
    list_subjects,
    load_eeg,
    load_fmri,
    load_audio_stimuli,
    load_events_for_run,
    epoch_eeg_by_stimulus,
)

# List available subjects
subject_dirs = list_subjects(DATA_DIR)
print(f"Found {len(subject_dirs)} subjects: {[s.name for s in subject_dirs[:5]]} ...")

In [ ]:
# Load EEG for subject 1, run 1
subject_dir = subject_dirs[0] if subject_dirs else DATA_DIR / "sub-01"

try:
    raw = load_eeg(subject_dir, run=1)
    print(f"Loaded: {raw}")
    print(f"Channels: {len(raw.ch_names)}")
    print(f"Sample rate: {raw.info['sfreq']} Hz")
    print(f"Duration: {raw.times[-1]:.1f} s")
except FileNotFoundError as e:
    print(f"Data not found ({e}).  Using synthetic data for demo.")
    raw = None

In [ ]:
# Load audio stimuli
try:
    stimuli = load_audio_stimuli(DATA_DIR, target_sr=1000)
    print(f"Loaded {len(stimuli)} audio stimuli")
    if stimuli:
        name, wav = next(iter(stimuli.items()))
        print(f"Example: '{name}', {len(wav)/1000:.1f} s at 1000 Hz")
except Exception as e:
    print(f"Audio not found: {e}")
    # Create synthetic stimuli for demo
    t = np.linspace(0, 40, 40000)
    stimuli = {f"piece_{i:02d}": np.sin(2*np.pi*(220 + i*20)*t).astype(np.float32)
               for i in range(36)}
    print(f"Using {len(stimuli)} synthetic stimuli")

In [ ]:
# Visualise a stimulus waveform
name, wav = next(iter(stimuli.items()))
t_wav = np.arange(len(wav)) / 1000.0

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].plot(t_wav[:5000], wav[:5000], lw=0.7, color='steelblue')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Waveform: {name}')

from scipy.signal import spectrogram
f, t_s, Sxx = spectrogram(wav, fs=1000, nperseg=256, noverlap=128)
axes[1].pcolormesh(t_s, f[:64], 10*np.log10(Sxx[:64]+1e-10), shading='gouraud', cmap='inferno')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_title('Spectrogram')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'stimulus_example.png', dpi=120)
plt.show()

## 2. EEG Preprocessing

In [ ]:
from src.eeg_preprocessing import preprocess_eeg

if raw is not None:
    print("Running preprocessing pipeline...")
    raw_clean, ica = preprocess_eeg(
        raw,
        l_freq=1.0,
        h_freq=40.0,
        target_sfreq=1000.0,
        n_ica_components=20,
        verbose=False,
    )
    print(f"Cleaned EEG: {raw_clean.info['sfreq']} Hz, {len(raw_clean.ch_names)} channels")
    print(f"ICA excluded components: {ica.exclude}")
else:
    # Synthetic EEG for demo
    n_channels, sfreq, duration = 31, 1000, 130
    n_samples = int(sfreq * duration)
    rng = np.random.default_rng(42)
    data = rng.standard_normal((n_channels, n_samples)) * 1e-6
    ch_names = [f'EEG{i:03d}' for i in range(1, n_channels+1)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw_clean = mne.io.RawArray(data, info, verbose=False)
    print("Using synthetic EEG data")

In [ ]:
# Plot power spectral density before/after preprocessing
if raw is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    raw.plot_psd(fmax=50, axes=axes[0], show=False)
    axes[0].set_title('PSD — Raw')
    raw_clean.plot_psd(fmax=50, axes=axes[1], show=False)
    axes[1].set_title('PSD — Cleaned (post ICA)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'psd_comparison.png', dpi=120)
    plt.show()
else:
    print("(Skipping PSD plot — no real data loaded)")

## 3. fMRI GLM Analysis and Dipole Selection

The fMRI GLM identifies music-responsive brain voxels.  We then greedily select
4 spatially distinct peak voxels (minimum 3 cm apart) as dipole locations.

In [ ]:
from src.fmri_analysis import (
    run_full_fmri_pipeline,
    select_dipole_locations,
    threshold_t_map,
)

# Check if fMRI data is available
fmri_available = (DATA_DIR / "sub-01" / "func").exists() if DATA_DIR.exists() else False

if fmri_available:
    print("Running fMRI GLM pipeline...")
    dipole_coords, group_t_map, subject_t_maps = run_full_fmri_pipeline(
        dataset_dir=DATA_DIR,
        subject_dirs=subject_dirs[:3],  # Use 3 subjects for demo
        n_runs=3,
        t_r=1.5,
        verbose=True,
    )
else:
    # Use realistic example dipole locations from the literature
    # (auditory cortex, supplementary motor area, etc.)
    print("fMRI data not found — using example dipole locations")
    dipole_coords = np.array([
        [-48,  -26,   8],   # Left primary auditory cortex (Heschl's gyrus)
        [ 52,  -22,  10],   # Right primary auditory cortex
        [ -4,  -10,  58],   # Supplementary motor area / premotor
        [  0,  -60,  20],   # Precuneus / posterior cingulate
    ], dtype=float)

print("\nSelected dipole locations (MNI mm):")
for i, d in enumerate(dipole_coords):
    print(f"  Dipole {i+1}: {d}")

np.save(OUTPUT_DIR / 'dipole_locations.npy', dipole_coords)

In [ ]:
# Visualise dipole locations on a brain surface (requires nilearn)
if fmri_available:
    try:
        from nilearn import plotting
        fig = plt.figure(figsize=(12, 4))
        display = plotting.plot_stat_map(
            group_t_map,
            threshold=3.0,
            display_mode='ortho',
            figure=fig,
            title='Group t-map (music > rest) with dipole locations',
        )
        display.add_markers(dipole_coords, marker_color='r', marker_size=50)
        plt.savefig(OUTPUT_DIR / 'dipole_locations.png', dpi=120)
        plt.show()
    except Exception as e:
        print(f"Visualisation error: {e}")
else:
    # Plot dipole locations in 3D MNI space
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(dipole_coords[:,0], dipole_coords[:,1], dipole_coords[:,2],
               s=200, c='red', marker='*', zorder=5)
    for i, d in enumerate(dipole_coords):
        ax.text(d[0]+2, d[1]+2, d[2]+2, f'D{i+1}', fontsize=10)
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')
    ax.set_zlabel('z (mm)')
    ax.set_title('Dipole locations (MNI space)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'dipole_locations_3d.png', dpi=120)
    plt.show()

## 4. Source Localisation and Feature Extraction

Build a spherical head model and apply eLoreta at the 4 dipole locations
to produce a 124-feature matrix (4 sources × 31 channels).

In [ ]:
from src.source_localization import build_spherical_forward, extract_features_from_epochs

# Create synthetic epochs for demonstration when real data is unavailable
def make_synthetic_epochs(n_epochs=36, n_channels=31, sfreq=1000, duration=40):
    """Generate synthetic EEG epochs for testing."""
    n_samples = int(sfreq * duration)
    rng = np.random.default_rng(0)
    data = rng.standard_normal((n_epochs, n_channels, n_samples)) * 1e-6
    ch_names = [f'EEG{i:03d}' for i in range(1, n_channels+1)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    # Add standard montage positions
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        # Use only channels in the montage
        available = [c for c in ch_names if c in montage.ch_names]
    except Exception:
        pass
    events = np.column_stack([
        np.arange(n_epochs) * n_samples,
        np.zeros(n_epochs, dtype=int),
        np.ones(n_epochs, dtype=int),
    ])
    raw_arr = mne.io.RawArray(
        np.concatenate([data[i] for i in range(n_epochs)], axis=1), info, verbose=False
    )
    epochs = mne.Epochs(raw_arr, events, event_id=1, tmin=0, tmax=duration-1/sfreq,
                        baseline=None, preload=True, verbose=False)
    return epochs

if raw is not None:
    events_df = load_events_for_run(subject_dir, run=1)
    if not events_df.empty:
        epochs = epoch_eeg_by_stimulus(raw_clean, events_df, tmin=0, tmax=40)
    else:
        print("No events found; using synthetic epochs")
        epochs = make_synthetic_epochs()
else:
    print("Using synthetic epochs for demonstration")
    epochs = make_synthetic_epochs(n_epochs=12, n_channels=31, sfreq=1000, duration=10)

print(f"Epochs: {epochs}")

In [ ]:
# Build forward model and extract features
print("Building spherical forward model...")
try:
    fwd = build_spherical_forward(epochs.info, dipole_coords)
    print(f"Forward model: {fwd}")

    print("Extracting features (eLoreta source projections)...")
    features = extract_features_from_epochs(
        epochs, fwd, target_sfreq=100.0, verbose=True
    )
    print(f"Feature matrix shape: {features.shape}")
    print(f"  (n_epochs={features.shape[0]}, n_features={features.shape[1]}, n_time={features.shape[2]})")
except Exception as e:
    print(f"Source localisation failed: {e}")
    print("Creating synthetic feature matrix for demo...")
    n_epochs = len(epochs)
    n_time = int(epochs.times[-1] * 100)  # at 100 Hz
    rng = np.random.default_rng(1)
    features = rng.standard_normal((n_epochs, 124, n_time)).astype(np.float32)
    print(f"Synthetic feature matrix: {features.shape}")

In [ ]:
# Visualise the feature matrix for one trial
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Feature heatmap
im = axes[0].imshow(features[0], aspect='auto', origin='lower',
                    cmap='RdBu_r', interpolation='nearest')
axes[0].set_xlabel('Time samples (100 Hz)')
axes[0].set_ylabel('Feature index (4 dipoles × 31 channels)')
axes[0].set_title('Feature matrix — Trial 0')
plt.colorbar(im, ax=axes[0])

# Draw dipole boundaries
for d in range(1, 4):
    axes[0].axhline(d * 31 - 0.5, color='yellow', lw=1, alpha=0.7)
    axes[0].text(features.shape[2]*0.02, d*31 - 16, f'D{d}', color='yellow', fontsize=9)

# Feature variance across trials
var = np.var(features, axis=(0, 2))  # (124,)
axes[1].plot(var, lw=1.2, color='darkorange')
for d in range(1, 4):
    axes[1].axvline(d * 31, color='grey', ls='--', lw=0.8)
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('Variance across trials')
axes[1].set_title('Feature variance')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_matrix.png', dpi=120)
plt.show()

## 5. biLSTM Model Training

Train the 4-layer bidirectional LSTM (250 hidden units) to reconstruct
the audio waveform from the 124-feature EEG representation.

In [ ]:
import torch
from src.models import BiLSTMDecoder, train_model, predict, count_parameters

# Model summary
model = BiLSTMDecoder(
    input_size=124,
    hidden_size=250,
    num_layers=4,
    output_size=1,
    dropout=0.0,
)
n_params = count_parameters(model)
print(f"Model architecture:")
print(model)
print(f"\nTrainable parameters: {n_params:,}")

In [ ]:
# Prepare training data
# features shape: (n_epochs, 124, n_time) → transpose to (n_epochs, n_time, 124)
X = np.transpose(features, (0, 2, 1)).astype(np.float32)
print(f"X shape (biLSTM input): {X.shape}")

# Prepare audio targets at 100 Hz
n_time = X.shape[1]
stim_keys = list(stimuli.keys())
y_list = []
for i in range(len(X)):
    audio_key = stim_keys[i % len(stim_keys)]
    audio = stimuli[audio_key][::10]  # downsample 1000 → 100 Hz
    if len(audio) >= n_time:
        y_i = audio[:n_time]
    else:
        y_i = np.pad(audio, (0, n_time - len(audio)))
    # Normalise
    m = np.abs(y_i).max()
    if m > 1e-8:
        y_i = y_i / m
    y_list.append(y_i.astype(np.float32))
y = np.stack(y_list)  # (n_epochs, n_time)
print(f"y shape (audio target): {y.shape}")

# Simple train/test split (last 20% as test)
n_train = int(0.8 * len(X))
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]
print(f"Train: {X_train.shape[0]} trials  |  Test: {X_test.shape[0]} trials")

In [ ]:
# Train the model (using a small number of epochs for the demo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

model_fresh = BiLSTMDecoder(input_size=124, hidden_size=250, num_layers=4)

trained_model, history = train_model(
    model_fresh,
    X_train,
    y_train,
    n_epochs=20,        # Use 100 for full run
    batch_size=8,
    lr=1e-3,
    device=device,
    verbose=True,
)

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], lw=2, color='teal')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_loss.png', dpi=120)
plt.show()

## 6. Evaluation

Evaluate using all four metrics from the paper:
- **r_time**: Pearson correlation in the time domain
- **r_freq**: Pearson correlation in the frequency domain  
- **SSIM**: Structural similarity of spectrograms
- **Rank accuracy**: % correct identification with bootstrap significance test

In [ ]:
from src.evaluation import evaluate_all, print_results_table, spectrogram_ssim

# Run inference on test set
preds = predict(trained_model, X_test, device=device)
print(f"Predictions shape: {preds.shape}")

# Full evaluation
results = evaluate_all(
    preds,
    y_test,
    fs=100.0,
    n_bootstrap=200,  # Use 4000 for full run
    verbose=True,
)
print_results_table(results)

In [ ]:
# Visualise predicted vs. target for the first test trial
trial_idx = 0
pred_ex = preds[trial_idx]
tgt_ex = y_test[trial_idx]
t_axis = np.arange(len(pred_ex)) / 100.0

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 2, figure=fig)

# Waveform comparison
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(t_axis, tgt_ex, lw=1.2, color='steelblue', label='Target', alpha=0.8)
ax1.plot(t_axis, pred_ex, lw=1.2, color='tomato', label='Decoded', alpha=0.8)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Waveform comparison (decoded vs. target)')
ax1.legend()

# Spectrogram comparison
from scipy.signal import spectrogram as spec
nperseg, noverlap = 32, 16
_, _, S_tgt = spec(tgt_ex, fs=100, nperseg=nperseg, noverlap=noverlap)
_, _, S_pred = spec(pred_ex, fs=100, nperseg=nperseg, noverlap=noverlap)

ax2 = fig.add_subplot(gs[1, 0])
ax2.imshow(10*np.log10(S_tgt+1e-10), aspect='auto', origin='lower',
           cmap='inferno', extent=[0, len(tgt_ex)/100, 0, 50])
ax2.set_title('Target spectrogram')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Frequency (Hz)')

ax3 = fig.add_subplot(gs[1, 1])
ax3.imshow(10*np.log10(S_pred+1e-10), aspect='auto', origin='lower',
           cmap='inferno', extent=[0, len(pred_ex)/100, 0, 50])
ax3.set_title('Decoded spectrogram')
ax3.set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reconstruction_example.png', dpi=120)
plt.show()

from scipy.stats import pearsonr
r_t, p_t = pearsonr(tgt_ex, pred_ex)
print(f"r_time = {r_t:.4f}  (p = {p_t:.4f})")

In [ ]:
# Rank accuracy visualisation
from src.evaluation import rank_accuracy

rank_acc, ranks = rank_accuracy(preds, y_test, metric='correlation')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(len(ranks)), ranks, color='steelblue', alpha=0.8)
axes[0].axhline(1, color='green', ls='--', lw=2, label='Rank 1 (perfect)')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Rank of correct stimulus')
axes[0].set_title(f'Rank accuracy per trial\n(mean = {rank_acc:.1f}%)')
axes[0].legend()

# Null distribution (bootstrap)
if 'null_dist' in results:
    axes[1].hist(results['null_dist'], bins=30, color='grey', alpha=0.7, label='Null (shuffled)')
    axes[1].axvline(results['rank_acc'], color='red', lw=2, label=f'Observed ({results["rank_acc"]:.1f}%)')
    axes[1].set_xlabel('Rank accuracy (%)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Bootstrap null distribution (p = {results["rank_acc_p"]:.4f})')
    axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rank_accuracy.png', dpi=120)
plt.show()

print(f"Rank accuracy = {rank_acc:.2f}% (bootstrap p = {results.get('rank_acc_p', 'N/A')})")

## 7. Cross-Fold Summary

The full pipeline uses 3-fold (leave-one-run-out) cross-validation. Here we
illustrate the expected results table structure.

In [ ]:
import pandas as pd

# Example results table (fill with your actual run results)
# For reference, approximate values from Daly (2023)
summary_data = {
    'Metric': ['r_time', 'r_freq', 'SSIM', 'Rank accuracy (%)'],
    'Paper (EEG-fMRI)': ['>0.0 (sig)', '>0.0 (sig)', '>0.0 (sig)', '>50%'],
    'This run (demo)': [
        f"{results.get('r_time_mean', 0):.4f} ± {results.get('r_time_std', 0):.4f}",
        f"{results.get('r_freq_mean', 0):.4f} ± {results.get('r_freq_std', 0):.4f}",
        f"{results.get('ssim_mean', 0):.4f} ± {results.get('ssim_std', 0):.4f}",
        f"{results.get('rank_acc', 0):.2f}%  (p={results.get('rank_acc_p', 1):.4f})",
    ]
}
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

In [ ]:
# Save all outputs
np.save(OUTPUT_DIR / 'demo_predictions.npy', preds)
np.save(OUTPUT_DIR / 'demo_targets.npy', y_test)
np.save(OUTPUT_DIR / 'demo_features.npy', features)

print("Demo outputs saved to:")
for f in sorted(OUTPUT_DIR.glob("*.png")) + sorted(OUTPUT_DIR.glob("*.npy")):
    print(f"  {f}")

## 8. Next Steps

To reproduce the full results from Daly (2023):

1. **Download the data**:
   ```
   python scripts/download_data.py --dataset ds002691 --output data/ds002691
   ```

2. **Run the full EEG-fMRI pipeline** (all 18 subjects, 3 folds each):
   ```
   python scripts/run_eegfmri.py --data_dir data/ds002691 --output_dir results/eegfmri --n_subjects 18
   ```

3. **Run the EEG-only pipeline** (19 subjects, using group-averaged dipoles):
   ```
   python scripts/run_eegonly.py \
       --data_dir data/ds002137 \
       --dipoles_file results/eegfmri/dipole_locations.npy \
       --output_dir results/eegonly
   ```

Key hyperparameters from the paper:
- biLSTM: 4 layers, 250 hidden units, input size 124
- Feature downsampling: 1000 Hz → 100 Hz (factor 10)
- Cross-validation: 3-fold leave-one-run-out per subject
- Bootstrap significance: 4000 shuffles
- Minimum dipole distance: 3 cm
- Head model grid: 1.5 × 1.5 × 1.5 cm